# 🚀 YOLOv12n Training - Construction Safety Dataset (3-Class, 960px)

**Optimized for H100 GPU (Colab Pro+) - Fixed Class Imbalance**

---

## 📊 Dataset Info:
- **Classes:** 3 total (merged from 18)
  - **Class 0:** Person
  - **Class 1:** Vehicle (Dump truck, Mixer, Tanker, Truck, Gazelle, Autocran)
  - **Class 2:** Equipment (Excavator, Roller, Bulldozer, Forklift, Crane, etc.)
- **Images:** ~12K train, ~2K validation (balanced, 65% person-only removed)
- **Size:** ~12-15GB
- **Training Resolution:** 960px (for small objects)
- **Deploy Resolution:** 640px (for Jetson Nano)

## ⚡ Expected Training Time:
- **H100 GPU:** 6-8 hours (100 epochs at 960px)
- **A100 GPU:** 10-12 hours
- **V100 GPU:** 16-20 hours

## 💰 Cost:
- **Colab Pro+:** $50/month (recommended for H100)
- **Effective cost:** $4-6 for this training

## 🎯 Expected Results:
- **mAP@50:** 0.70-0.80 (vs 0.45 with 18 classes)
- **mAP@50-95:** 0.50-0.60 (vs 0.35 with 18 classes)
- **Improvement:** +56% better detection!

---

## 📋 Prerequisites:
1. ✅ Dataset uploaded to Google Drive: `YOLOv12_Training/merged_construction_safety_3class_balanced.zip`
2. ✅ Colab Pro+ subscription active
3. ✅ H100 GPU selected in runtime

---

**Run cells in order** ⬇️

## 🔧 Cell 1: Check GPU & Environment

Verify you have H100 GPU access

In [ ]:
!nvidia-smi

import torch
print(f"\n{'='*70}")
print("GPU Information")
print(f"{'='*70}")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    print(f"GPU device: {torch.cuda.get_device_name(0)}")
    print(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    
    # Check if it's H100
    gpu_name = torch.cuda.get_device_name(0)
    if 'H100' in gpu_name:
        print("\n✅ H100 GPU detected! Training will be super fast (~3-4 hours)")
    elif 'A100' in gpu_name:
        print("\n✅ A100 GPU detected! Training will be fast (~5-6 hours)")
    elif 'V100' in gpu_name:
        print("\n⚠️  V100 GPU detected. Training will take ~8-10 hours")
    elif 'T4' in gpu_name:
        print("\n⚠️  T4 GPU detected. Consider upgrading to Colab Pro for H100/A100")
        print("   Training will take ~12-18 hours on T4")
else:
    print("\n❌ No GPU detected! Go to Runtime → Change runtime type → GPU")
print(f"{'='*70}\n")

## 📁 Cell 2: Mount Google Drive

Connect to your Google Drive where dataset is stored

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
print("\n✅ Google Drive mounted successfully!")
print(f"\nChecking for dataset...")

# Check if dataset exists
dataset_zip = '/content/drive/MyDrive/YOLOv12_Training/merged_construction_safety_3class_balanced.zip'
if os.path.exists(dataset_zip):
    size_gb = os.path.getsize(dataset_zip) / 1e9
    print(f"✅ Dataset found: {dataset_zip}")
    print(f"   Size: {size_gb:.2f} GB")
else:
    print(f"❌ Dataset not found at: {dataset_zip}")
    print("\nPlease upload merged_construction_safety_3class_balanced.zip to:")
    print("   Google Drive → YOLOv12_Training/")
    print("\nOr adjust the path above if you uploaded it elsewhere")

## 📦 Cell 3: Install Dependencies

Install Ultralytics YOLO and required packages

In [ ]:
!pip install -q ultralytics>=8.1.0
!pip install -q tensorboard

print("\n✅ Dependencies installed!")

# Verify installation
from ultralytics import YOLO
print(f"Ultralytics version: {YOLO.__version__ if hasattr(YOLO, '__version__') else 'installed'}")

## 📂 Cell 4: Extract Dataset

Unzip dataset to Colab runtime (faster I/O than Drive)

In [ ]:
import zipfile
from tqdm import tqdm

# Paths
dataset_zip = '/content/drive/MyDrive/YOLOv12_Training/merged_construction_safety_3class_balanced.zip'
extract_to = '/content/merged_construction_safety_3class_balanced'

print(f"{'='*70}")
print("Extracting Dataset to Colab Runtime")
print(f"{'='*70}")
print(f"From: {dataset_zip}")
print(f"To: {extract_to}")
print(f"\nThis will take ~3-5 minutes...\n")

# Extract with progress bar
with zipfile.ZipFile(dataset_zip, 'r') as zip_ref:
    members = zip_ref.namelist()
    for member in tqdm(members, desc="Extracting"):
        zip_ref.extract(member, '/content/')

print(f"\n✅ Dataset extracted successfully!")
print(f"\nDataset location: {extract_to}")

# Verify structure
!ls -lh /content/merged_construction_safety_3class_balanced/

## ✅ Cell 5: Verify Dataset

Check dataset structure and integrity

In [ ]:
import yaml
from pathlib import Path

# Load data.yaml
data_yaml_path = '/content/merged_construction_safety_3class_balanced/data.yaml'

with open(data_yaml_path, 'r') as f:
    data_config = yaml.safe_load(f)

print(f"{'='*70}")
print("Dataset Configuration")
print(f"{'='*70}")
print(f"Path: {data_config['path']}")
print(f"Train: {data_config['train']}")
print(f"Val: {data_config['val']}")
print(f"Classes: {data_config['nc']}")
print(f"\nClass Names:")
for idx, name in data_config['names'].items():
    print(f"  {idx}: {name}")

# Count images
train_images = list(Path('/content/merged_construction_safety_3class_balanced/train/images').glob('*.jpg'))
val_images = list(Path('/content/merged_construction_safety_3class_balanced/valid/images').glob('*.jpg'))
train_labels = list(Path('/content/merged_construction_safety_3class_balanced/train/labels').glob('*.txt'))
val_labels = list(Path('/content/merged_construction_safety_3class_balanced/valid/labels').glob('*.txt'))

print(f"\n{'='*70}")
print("Dataset Statistics")
print(f"{'='*70}")
print(f"Train images: {len(train_images):,}")
print(f"Train labels: {len(train_labels):,}")
print(f"Val images: {len(val_images):,}")
print(f"Val labels: {len(val_labels):,}")
print(f"\nTotal: {len(train_images) + len(val_images):,} images")

if len(train_images) == len(train_labels) and len(val_images) == len(val_labels):
    print("\n✅ Dataset verified - all images have corresponding labels!")
else:
    print("\n⚠️  Mismatch between images and labels")

print(f"\n{'='*70}")
print("🎯 3-Class Balanced Dataset")
print(f"{'='*70}")
print("✅ Class 0: Person")
print("✅ Class 1: Vehicle (merged 6 vehicle types)")
print("✅ Class 2: Equipment (merged 11 equipment types)")
print("✅ Person-only images downsampled by 65%")
print("✅ Training resolution: 960px (for small objects)")
print(f"{'='*70}\n")

## 🏋️ Cell 6: Train YOLOv12n (960px for Small Objects)

**This is the main training cell - will run for ~6-8 hours on H100**

Parameters optimized for 960px training:
- **Image size: 960px** (CRITICAL for tiny objects!)
- **Batch size: 64** (reduced from 128 due to larger images)
- **Classes: 3** (person, vehicle, equipment)
- **Cache: disk** (faster for large datasets)
- **Workers: 8**

**Why 960px?**
- Your objects are tiny (8x8 pixels at 640px)
- Training at 960px allows model to learn features
- Export at 640px for Jetson Nano deployment
- Model "remembers" high-res features even at lower resolution

In [ ]:
from ultralytics import YOLO
import torch
import yaml

# Create output directory in Drive for persistence
!mkdir -p /content/drive/MyDrive/YOLOv12_Training/runs

# Fix data.yaml path if needed
data_yaml_path = '/content/merged_construction_safety_3class_balanced/data.yaml'
with open(data_yaml_path, 'r') as f:
    data_config = yaml.safe_load(f)

# Check and fix path
if data_config['path'] != '/content/merged_construction_safety_3class_balanced':
    print(f"⚠️  Incorrect 'path' found in data.yaml: {data_config['path']}")
    data_config['path'] = '/content/merged_construction_safety_3class_balanced'
    with open(data_yaml_path, 'w') as f:
        yaml.dump(data_config, f)
    print(f"✅ Correcting 'path' in {data_yaml_path} to: /content/merged_construction_safety_3class_balanced")

# Check GPU memory
if torch.cuda.is_available():
    gpu_memory_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU Memory: {gpu_memory_gb:.1f} GB\n")
    
    # Optimize batch size based on GPU (960px needs more VRAM!)
    if gpu_memory_gb >= 75:  # H100 has ~80GB
        batch_size = 64  # REDUCED from 128 due to 960px
        print("✅ H100 detected - using batch size 64 (960px training)")
    elif gpu_memory_gb >= 35:  # A100 has ~40GB
        batch_size = 32  # REDUCED from 64
        print("✅ A100 detected - using batch size 32 (960px training)")
    elif gpu_memory_gb >= 14:  # V100 has ~16GB
        batch_size = 16
        print("✅ V100 detected - using batch size 16 (960px training)")
    else:  # T4 has ~16GB
        batch_size = 8
        print("⚠️  T4 detected - using batch size 8 (may be slow)")
else:
    batch_size = 4
    print("⚠️  No GPU - using batch size 4 (very slow)")

print(f"\n{'='*70}")
print("Starting YOLOv12n Training (3-Class, 960px)")
print(f"{'='*70}")
print(f"Batch Size: {batch_size}")
print(f"Image Size: 960px (for small objects!)")
print(f"Epochs: 100")
print(f"Dataset: merged_construction_safety_3class_balanced (3 classes)")
print(f"Classes: 0=Person, 1=Vehicle, 2=Equipment")
print(f"{'='*70}\n")

# Initialize model
model = YOLO('yolo12n.pt')  # Auto-download pretrained weights

# Training parameters (960px optimized for small objects)
results = model.train(
    data='/content/merged_construction_safety_3class_balanced/data.yaml',
    epochs=100,
    imgsz=960,  # CHANGED FROM 640 → 960 (FOR SMALL OBJECTS!)
    batch=batch_size,
    device=0,  # GPU 0
    project='/content/drive/MyDrive/YOLOv12_Training/runs',  # Save to Drive
    name='yolo12n_construction_3class_960px',  # NEW NAME
    patience=50,
    save=True,
    save_period=10,  # Save checkpoint every 10 epochs
    cache='disk',  # Cache to disk (faster than RAM for large datasets)
    workers=8,
    optimizer='AdamW',
    verbose=True,
    seed=42,
    deterministic=False,
    cos_lr=True,
    close_mosaic=10,
    resume=False,
    amp=True,  # Automatic Mixed Precision (FP16)
    fraction=1.0,  # Use 100% of data
    lr0=0.01,
    lrf=0.01,
    momentum=0.937,
    weight_decay=0.0005,
    warmup_epochs=3.0,
    box=7.5,
    cls=0.5,
    dfl=1.5,
    val=True,
    plots=True,
)

print(f"\n{'='*70}")
print("✅ Training Complete!")
print(f"{'='*70}")
print(f"\nBest model: /content/drive/MyDrive/YOLOv12_Training/runs/yolo12n_construction_3class_960px/weights/best.pt")
print(f"Last model: /content/drive/MyDrive/YOLOv12_Training/runs/yolo12n_construction_3class_960px/weights/last.pt")
print(f"\n🎯 Expected mAP@50: 0.70-0.80 (vs 0.45 with 18 classes)")
print(f"🎯 Expected mAP@50-95: 0.50-0.60 (vs 0.35 with 18 classes)")

## 📊 Cell 7: View Training Results

Display training metrics and plots

In [ ]:
from IPython.display import Image, display
import pandas as pd

results_dir = '/content/drive/MyDrive/YOLOv12_Training/runs/yolo12n_construction_3class_960px'

print(f"{'='*70}")
print("Training Results")
print(f"{'='*70}\n")

# Load results CSV
results_csv = f"{results_dir}/results.csv"
df = pd.read_csv(results_csv)

# Display final metrics
print("Final Metrics (last epoch):")
print(df.tail(1).to_string())

# Display best metrics
print(f"\nBest mAP@50: {df['metrics/mAP50(B)'].max():.4f}")
print(f"Best mAP@50-95: {df['metrics/mAP50-95(B)'].max():.4f}")

# Display training plots
print(f"\n{'='*70}")
print("Training Curves")
print(f"{'='*70}\n")

results_img = f"{results_dir}/results.png"
display(Image(filename=results_img, width=800))

print(f"\n{'='*70}")
print("Confusion Matrix")
print(f"{'='*70}\n")

confusion_img = f"{results_dir}/confusion_matrix.png"
display(Image(filename=confusion_img, width=800))

## 💾 Cell 8: Download Trained Model

Download best.pt to your local machine

In [ ]:
from google.colab import files

best_model = '/content/drive/MyDrive/YOLOv12_Training/runs/yolo12n_construction_3class_960px/weights/best.pt'

print(f"{'='*70}")
print("Download Trained Model")
print(f"{'='*70}")
print(f"\nModel: {best_model}")
print(f"Training resolution: 960px")
print(f"Classes: 3 (person, vehicle, equipment)")
print(f"\nDownloading...\n")

files.download(best_model)

print("\n✅ Model downloaded!")
print("\nSave this as: yolo12n_construction_3class_960px.pt")
print("\n📝 IMPORTANT: Export to 640px for Jetson Nano deployment:")
print("\n# In your deployment environment:")
print("from ultralytics import YOLO")
print("model = YOLO('yolo12n_construction_3class_960px.pt')")
print("model.export(format='onnx', imgsz=640)  # Export at 640px for Jetson")
print("\nModel was trained at 960px but can be deployed at 640px!")
print("It will remember the high-resolution features. 🎯")

## 🧪 Cell 9: Test Inference (Optional)

Test the trained model on a sample image

In [ ]:
from ultralytics import YOLO
from IPython.display import Image, display

# Load trained model
model = YOLO('/content/drive/MyDrive/YOLOv12_Training/runs/yolo12n_construction_3class_960px/weights/best.pt')

# Get a sample image from validation set
import random
from pathlib import Path

val_images = list(Path('/content/merged_construction_safety_3class_balanced/valid/images').glob('*.jpg'))
sample_image = random.choice(val_images)

print(f"Running inference on: {sample_image.name}\n")

# Run prediction
results = model.predict(source=str(sample_image), save=True, conf=0.25)

# Display result
result_img = results[0].path.replace('.jpg', '_pred.jpg')
print(f"\nResult saved to: {result_img}")
display(Image(filename=result_img, width=800))

# Print detections
print(f"\nDetections:")
class_names = {0: 'Person', 1: 'Vehicle', 2: 'Equipment'}
for box in results[0].boxes:
    cls = int(box.cls[0])
    conf = float(box.conf[0])
    class_name = class_names.get(cls, f'Unknown({cls})')
    print(f"  {class_name}: {conf:.2f}")

## 📈 Cell 10: TensorBoard (Optional)

Launch TensorBoard to monitor training in real-time

In [ ]:
%load_ext tensorboard
%tensorboard --logdir /content/drive/MyDrive/YOLOv12_Training/runs/yolo12n_construction_3class_960px

---

## ✅ Training Complete!

### 📦 What You Have:
- ✅ Trained YOLOv12n model (3-class, 960px): `best.pt`
- ✅ Training metrics: `results.csv`
- ✅ Training curves: `results.png`
- ✅ Confusion matrix: `confusion_matrix.png`
- ✅ All saved in Google Drive (persistent)

### 🎯 Model Details:
- **Classes:** 3 (person, vehicle, equipment)
- **Training resolution:** 960px
- **Deploy resolution:** 640px (for Jetson Nano)
- **Expected mAP@50:** 0.70-0.80 (+56% vs 18-class model!)
- **Expected mAP@50-95:** 0.50-0.60 (+43% vs 18-class model!)

### 🚀 Next Steps:

#### 1. Download `best.pt` to your Mac
```bash
# Model already downloaded in Cell 8
```

#### 2. Export for Jetson Nano (640px deployment)
```python
from ultralytics import YOLO

# Load 960px trained model
model = YOLO('yolo12n_construction_3class_960px.pt')

# Export to ONNX at 640px for Jetson
model.export(format='onnx', imgsz=640, dynamic=False)
# Creates: yolo12n_construction_3class_960px.onnx

# Also export to TensorRT for Jetson (optional)
model.export(format='engine', imgsz=640, device=0)
```

#### 3. Integrate with RT-MonoDepth pipeline
```python
# In realtime_depth_video.py
from ultralytics import YOLO

# Use 3-class model
yolo_model = YOLO('yolo12n_construction_3class_960px.onnx')

# Class mapping
class_names = {
    0: 'Person',
    1: 'Vehicle',  # Merged: Dump truck, Mixer, Tanker, Truck, etc.
    2: 'Equipment'  # Merged: Excavator, Roller, Bulldozer, etc.
}
```

#### 4. Optional: Two-Stage Classification
If you need to distinguish specific vehicle types:

```python
# Stage 1: YOLOv12n detects "vehicle" (fast, 30 FPS)
results = yolo_model(frame)

# Stage 2: For each vehicle detection, run lightweight classifier
for detection in results:
    if detection.class == 1:  # Vehicle
        crop = frame[detection.bbox]
        vehicle_type = vehicle_classifier(crop)  # ResNet18, MobileNet
        # Returns: "Dump truck", "Mixer", "Tanker", etc.
```

### 📝 Notes:
- Model was trained at 960px → learned tiny object features
- Export at 640px → fast inference on Jetson Nano
- Model "remembers" high-resolution features even at lower deployment resolution
- 3-class model is much more reliable than 18-class on limited hardware

### 🎯 Expected Performance:
- **Jetson Nano:** 25-30 FPS @ 640px
- **mAP@50:** 0.70-0.80 (excellent for construction safety)
- **Balanced detection:** Person, vehicles, equipment all detected equally well

---

**Created by:** RT-MonoDepth-Construction Project  
**Date:** December 12, 2025  
**GPU:** H100 (Colab Pro+)  
**Training:** 3-class, 960px (balanced dataset)  
**Deployment:** 640px (Jetson Nano compatible)